In [ ]:
def entropy(y, poids_classes):

    # Calculer les effectifs de chaque classe
    effectifs = y.value_counts().astype(float)

    # Calculer les effectifs pondérés
    effectifs_ponderes = effectifs.copy()

    for classe in effectifs.index:
        effectifs_ponderes.loc[classe] = (
            effectifs.loc[classe] * poids_classes[classe]
        )

    # Calculer les probabilités pondérées
    probabilites = effectifs_ponderes / effectifs_ponderes.sum()

    # Calculer l'entropie
    entropie = 0

    for p in probabilites:
        if p > 0:
            entropie -= p * np.log2(p)

    return entropie


In [ ]:
def poids_total(y, poids_classes):
    return y.map(poids_classes).sum()


In [ ]:
def information_gain(y, partitions, poids_classes):

    # Entropie pondérée avant la séparation
    entropie_parent = entropy(y, poids_classes)

    # Poids total du groupe parent
    poids_parent = poids_total(y, poids_classes)

    # Entropie pondérée après la séparation
    entropie_ponderee = 0

    for subset in partitions:

        if len(subset) == 0:
            continue

        # Calculer le poids relatif du sous-groupe
        poids = poids_total(subset, poids_classes) / poids_parent

        # Ajouter sa contribution à l'entropie
        entropie_ponderee += poids * entropy(
            subset, poids_classes
        )

    # Gain d'information
    gain = entropie_parent - entropie_ponderee

    return gain


In [ ]:
def best_split(X, y, categorical_features, numerical_features,
               poids_classes):

    meilleur_gain = -1
    meilleure_feature = None
    meilleure_valeur = None
    meilleur_type = None

    # Tester les variables catégorielles
    for feature in categorical_features:

        valeurs = X[feature].dropna().unique()

        for valeur in valeurs:

            masque_gauche = X[feature] == valeur
            masque_droit = X[feature] != valeur

            gauche_y = y[masque_gauche]
            droit_y = y[masque_droit]

            gain = information_gain(
                y,
                [gauche_y, droit_y],
                poids_classes
            )

            if gain > meilleur_gain:
                meilleur_gain = gain
                meilleure_feature = feature
                meilleure_valeur = valeur
                meilleur_type = "categorical"

    # Tester les variables numériques
    for feature in numerical_features:

        valeurs = np.sort(X[feature].dropna().unique())

        # Calculer les seuils possibles
        seuils = (valeurs[:-1] + valeurs[1:]) / 2

        for seuil in seuils:

            masque_gauche = X[feature] <= seuil
            masque_droit = X[feature] > seuil

            gauche_y = y[masque_gauche]
            droit_y = y[masque_droit]

            gain = information_gain(
                y,
                [gauche_y, droit_y],
                poids_classes
            )

            if gain > meilleur_gain:
                meilleur_gain = gain
                meilleure_feature = feature
                meilleure_valeur = seuil
                meilleur_type = "numerical"

    return (
        meilleure_feature,
        meilleure_valeur,
        meilleur_type,
        meilleur_gain
    )


In [ ]:
best_feature, split_value, split_type, gain = best_split(
    X_train_tree,
    y_train_tree,
    categorical_cols,
    numerical_cols,
    poids_classes
)

print("Best feature:", best_feature)
print("Split value:", split_value)
print("Split type:", split_type)
print("Information Gain:", gain)

In [ ]:
def majority_class(y, poids_classes):

    effectifs_ponderes = (
        y.map(poids_classes)
        .groupby(y)
        .sum()
    )

    return effectifs_ponderes.idxmax()


In [ ]:
def build_tree(X, y, categorical_features, numerical_features,
               poids_classes, depth=0, max_depth=5,
               min_samples_split=40, min_gain=0.001):

    # Déterminer la classe prédite par le nœud
    prediction = majority_class(y, poids_classes)

    # 1. Vérifier les critères d'arrêt

    # Arrêter si toutes les observations appartiennent à la même classe
    if y.nunique() == 1:
        return {"prediction": prediction}

    # Arrêter si la profondeur maximale est atteinte
    if depth >= max_depth:
        return {"prediction": prediction}

    # Arrêter si le nombre d'observations est insuffisant
    if len(y) < min_samples_split:
        return {"prediction": prediction}

    # 2. Rechercher la meilleure séparation
    feature, split_value, split_type, gain = best_split(
        X,
        y,
        categorical_features,
        numerical_features,
        poids_classes
    )

    # Arrêter si aucune séparation pertinente n'est trouvée
    if feature is None or gain <= min_gain:
        return {"prediction": prediction}

    # 3. Créer les deux branches

    if split_type == "numerical":
        left_mask = X[feature] <= split_value
        right_mask = X[feature] > split_value

    else:
        left_mask = X[feature] == split_value
        right_mask = X[feature] != split_value

    # Éviter de créer une branche vide
    if not left_mask.any() or not right_mask.any():
        return {"prediction": prediction}

    # 4. Construire récursivement les deux sous-arbres

    left_subtree = build_tree(
        X.loc[left_mask],
        y.loc[left_mask],
        categorical_features,
        numerical_features,
        poids_classes,
        depth + 1,
        max_depth,
        min_samples_split,
        min_gain
    )

    right_subtree = build_tree(
        X.loc[right_mask],
        y.loc[right_mask],
        categorical_features,
        numerical_features,
        poids_classes,
        depth + 1,
        max_depth,
        min_gain
    )

    # Retourner le nœud avec ses informations de séparation
    return {
        "feature": feature,
        "split_value": split_value,
        "split_type": split_type,
        "gain": gain,
        "n_samples": len(y),
        "left": left_subtree,
        "right": right_subtree,
        "prediction": prediction
    }


In [ ]:
# Construction de l'arbre de décision pondéré

arbre_pondere = build_tree(
    X_train_tree,
    y_train_tree,
    categorical_cols,
    numerical_cols,
    poids_classes,
    max_depth=5,
    min_samples_split=40,
    min_gain=0.001
)

print("Arbre de décision pondéré construit avec succès.")

In [ ]:
# Prédiction avec l'arbre pondéré
y_pred_pondere = predict(arbre_pondere, X_test_tree)

print("Premières prédictions :")
print(y_pred_pondere.head(10))

print("\nClasses réelles :")
print(y_test_tree.head(10))